In [ ]:
import pandas as pd
import json

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)

with open("data/raw/virginia_de.json") as f:
    jobs = json.load(f)

df = pd.DataFrame(jobs)
len(df)

In [ ]:
df.shape

In [ ]:
df.columns.to_list()

In [ ]:
df.head(3)

In [ ]:
df.iloc[0]

In [ ]:
df["remote"].value_counts(dropna=False)

In [ ]:
df["hybrid"].value_counts(dropna=False)

In [ ]:
pd.crosstab(df["remote"].fillna("missing"), df["hybrid"].fillna("missing"))

In [ ]:
[c for c in df.columns if any(w in c.lower() for w in ["site", "office", "remote", "hybrid", "work", "location", "type"])]

In [ ]:
df["work_arrangement"].value_counts(dropna=False)

In [ ]:
df["work_arrangement_source"].value_counts(dropna=False)

In [ ]:
import pathlib

frames = []
for f in pathlib.Path("data/raw").glob("*.json"):
    part = pd.DataFrame(json.loads(f.read_text()))
    part["slice"] = f.stem
    frames.append(part)

all_jobs = pd.concat(frames, ignore_index=True)
print(len(all_jobs), all_jobs["id"].nunique())

In [ ]:
membership = all_jobs[["id", "slice"]].drop_duplicates()
postings = all_jobs.drop(columns="slice").drop_duplicates(subset="id")

print(len(postings), len(membership))

In [ ]:
import numpy as np

text = postings["description"].fillna("").str.lower()
src = postings["work_arrangement"].str.lower()

conditions = [
    src.notna(),
    text.str.contains(r"fully remote|100% remote|remote position|this (?:role|position) is remote"),
    text.str.contains(r"\bhybrid\b|days? (?:a|per) week in (?:the )?office"),
    text.str.contains(r"on-?site|in[- ]office|in the office"),
]
choices = [src, "remote", "hybrid", "on-site"]

postings["arrangement"] = np.select(conditions, choices, default="unknown")
postings["arrangement"].value_counts()

In [ ]:
postings["arrangement"] = postings["arrangement"].replace({"onsite": "on-site"})

by_slice = membership.merge(postings[["id", "arrangement"]], on="id")
pd.crosstab(by_slice["slice"], by_slice["arrangement"], normalize="index").round(2)

In [ ]:
loc = postings["short_location"].fillna(postings["location"]).fillna("")
postings["in_va"] = loc.str.contains(r",\s*(?:VA|Virginia)\b") | loc.str.strip().eq("Charlottesville")

groups = {
    "local_de_towns":  "Local data engineering",
    "local_de_metros": "Local data engineering",
    "local_adjacent":  "Local related tech roles",
    "virginia_de":     "Virginia data engineering",
    "us_all_de":       "US data engineering",
}

g = membership.assign(group=membership["slice"].map(groups)).dropna(subset=["group"])
g = g.merge(postings[["id", "in_va", "arrangement"]], on="id")
g = g[~g["group"].str.startswith("Local") | g["in_va"]]   # local groups: Virginia only
g = g.drop_duplicates(subset=["id", "group"])             # towns + metros overlap

print(g["group"].value_counts())
pd.crosstab(g["group"], g["arrangement"], normalize="index").round(2)